In [2]:
import gdown
import pandas as pd
from pathlib import Path
import numpy as np
pd.set_option("display.max_columns", None)
from sklearn.preprocessing import StandardScaler
import holidays




In [ ]:
CLEAN_DIR = Path("../data/cleansed")
CLEAN_FILE = CLEAN_DIR / "all_flights_2018-2022_cleansed.parquet"
df = pd.read_parquet(CLEAN_FILE)


In [ ]:
df.head()

,FlightDate,Airline,Origin,Dest,CRSDepTime,CRSElapsedTime,Distance,Year,Quarter,Month,DayofMonth,DayOfWeek,Marketing_Airline_Network,Operated_or_Branded_Code_Share_Partners,DOT_ID_Marketing_Airline,IATA_Code_Marketing_Airline,Flight_Number_Marketing_Airline,Operating_Airline,DOT_ID_Operating_Airline,IATA_Code_Operating_Airline,Tail_Number,Flight_Number_Operating_Airline,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,OriginCityName,OriginState,OriginStateFips,OriginStateName,OriginWac,DestAirportID,DestAirportSeqID,DestCityMarketID,DestCityName,DestState,DestStateFips,DestStateName,DestWac,DepTimeBlk,CRSArrTime,ArrTimeBlk,DistanceGroup,year,target,date,dep_hour,tmpf,vsby,sknt,p01i,relh,gust,day_name,month_sin,month_cos
0,2018-01-04,Endeavor Air Inc.,ATL,ABY,1037,60.0,145.0,2018,1,1,4,Thursday,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N981EV,3298,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,1000-1059,1137,1100-1159,1,2018,On time,2018-01-04,10,24.10,10.000000,14.166667,0.00,54.360,21.0,Thursday,0.5,0.866025
1,2018-01-13,Endeavor Air Inc.,ATL,ABY,1037,59.0,145.0,2018,1,1,13,Saturday,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N8976E,3298,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,1000-1059,1136,1100-1159,1,2018,On time,2018-01-13,10,28.90,10.000000,17.200000,0.00,74.760,26.0,Saturday,0.5,0.866025
2,2018-01-17,Endeavor Air Inc.,ATL,ABY,1037,60.0,145.0,2018,1,1,17,Wednesday,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N8972E,3298,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,1000-1059,1137,1100-1159,1,2018,Cancelled,2018-01-17,10,17.46,0.661765,14.176471,0.39,90.992,19.0,Wednesday,0.5,0.866025
3,2018-01-30,Endeavor Air Inc.,ATL,ABY,1037,60.0,145.0,2018,1,1,30,Tuesday,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N8946A,3298,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,1000-1059,1137,1100-1159,1,2018,On time,2018-01-30,10,30.90,10.000000,16.461538,0.00,60.920,20.0,Tuesday,0.5,0.866025
4,2018-01-13,Endeavor Air Inc.,ATL,EVV,941,87.0,350.0,2018,1,1,13,Saturday,DL,DL_CODESHARE,19790,DL,3299,9E,20363,9E,N8928A,3299,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,11612,1161206,31612,"Evansville, IN",IN,18,Indiana,42,0900-0959,1008,1000-1059,2,2018,Delayed,2018-01-13,9,28.00,10.000000,15.600000,0.00,77.570,27.0,Saturday,0.5,0.866025


In [7]:
df['DayOfWeek'] = df['FlightDate'].dt.day_name()

In [17]:
day_map = {
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5,
    "Sunday": 6
}

df["DayOfWeek_num"] = df["DayOfWeek"].map(day_map)


In [18]:
df["month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)


In [19]:
df["dow_sin"] = np.sin(2 * np.pi * df["DayOfWeek_num"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["DayOfWeek_num"] / 7)

In [13]:
# Standardizing weather features
num_cols = [
    "Distance",
    "tmpf", "vsby", "sknt", "relh", "gust"
]

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])


In [14]:
# normalized distance from the mean of airport origin
df["distance_origin_norm"] = (
    df["Distance"] - df.groupby("Origin")["Distance"].transform("mean")
)


In [15]:
#has rain binary column
df["has_precip"] = (df["p01i"] > 0).astype(int)


In [23]:
# holidays
us_holidays = holidays.US()

# Is Holiday (0/1)
df["is_holiday"] = df["FlightDate"].dt.date.apply(
    lambda x: 1 if x in us_holidays else 0
)

# Within ±3 days of a holiday
def near_holiday(date, holiday_calendar, window=3):
    for offset in range(-window, window + 1):
        if (date + pd.Timedelta(days=offset)) in holiday_calendar:
            return 1
    return 0

df["near_holiday_3d"] = df["FlightDate"].dt.date.apply(
    lambda x: near_holiday(pd.Timestamp(x), us_holidays)
)


In [24]:
df.head()

,FlightDate,Airline,Origin,Dest,CRSDepTime,CRSElapsedTime,Distance,Year,Quarter,Month,DayofMonth,DayOfWeek,Marketing_Airline_Network,Operated_or_Branded_Code_Share_Partners,DOT_ID_Marketing_Airline,IATA_Code_Marketing_Airline,Flight_Number_Marketing_Airline,Operating_Airline,DOT_ID_Operating_Airline,IATA_Code_Operating_Airline,Tail_Number,Flight_Number_Operating_Airline,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,OriginCityName,OriginState,OriginStateFips,OriginStateName,OriginWac,DestAirportID,DestAirportSeqID,DestCityMarketID,DestCityName,DestState,DestStateFips,DestStateName,DestWac,DepTimeBlk,CRSArrTime,ArrTimeBlk,DistanceGroup,year,target,date,dep_hour,tmpf,vsby,sknt,p01i,relh,gust,day_name,month_sin,month_cos,distance_origin_norm,has_precip,DayOfWeek_num,dow_sin,dow_cos,is_holiday,near_holiday_3d
0,2018-01-04,Endeavor Air Inc.,ATL,ABY,1037,60.0,-1.177273,2018,1,1,4,Thursday,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N981EV,3298,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,1000-1059,1137,1100-1159,1,2018,On time,2018-01-04,10,-1.879350,0.342046,0.501569,0.00,-0.060152,0.030893,Thursday,0.5,0.866025,-0.856184,0,3,0.433884,-0.900969,0,1
1,2018-01-13,Endeavor Air Inc.,ATL,ABY,1037,59.0,-1.177273,2018,1,1,13,Saturday,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N8976E,3298,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,1000-1059,1136,1100-1159,1,2018,On time,2018-01-13,10,-1.642469,0.342046,1.243438,0.00,0.877135,0.961287,Saturday,0.5,0.866025,-0.856184,0,5,-0.974928,-0.222521,0,1
2,2018-01-17,Endeavor Air Inc.,ATL,ABY,1037,60.0,-1.177273,2018,1,1,17,Wednesday,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N8972E,3298,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,1000-1059,1137,1100-1159,1,2018,Cancelled,2018-01-17,10,-2.207036,-5.378799,0.503966,0.39,1.622922,-0.341265,Wednesday,0.5,0.866025,-0.856184,1,2,0.974928,-0.222521,0,1
3,2018-01-30,Endeavor Air Inc.,ATL,ABY,1037,60.0,-1.177273,2018,1,1,30,Tuesday,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N8946A,3298,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,1000-1059,1137,1100-1159,1,2018,On time,2018-01-30,10,-1.543769,0.342046,1.062831,0.00,0.241250,-0.155186,Tuesday,0.5,0.866025,-0.856184,0,1,0.781831,0.623490,0,0
4,2018-01-13,Endeavor Air Inc.,ATL,EVV,941,87.0,-0.824704,2018,1,1,13,Saturday,DL,DL_CODESHARE,19790,DL,3299,9E,20363,9E,N8928A,3299,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,11612,1161206,31612,"Evansville, IN",IN,18,Indiana,42,0900-0959,1008,1000-1059,2,2018,Delayed,2018-01-13,9,-1.686885,0.342046,0.852122,0.00,1.006242,1.147366,Saturday,0.5,0.866025,-0.503615,0,5,-0.974928,-0.222521,0,1


In [1]:
df[df.is_holiday == 1].head()

NameError: name 'df' is not defined